# Gemma-Cyber v0.2 (exp-002): Free Cloud QLoRA Training & GGUF Export

This notebook fine-tunes **Gemma-3-4B-it** on the curated `sft_v0.2.jsonl` cybersecurity dataset using **QLoRA** (4-bit quantization + LoRA adapters), merges the weights, and exports a quantized **GGUF** model for local inference with Ollama.

`sft_v0.2` is the first dataset with genuine answer-level diversity (277/277 unique answers vs. v0.1's 91/360) across 15 task types, and explicitly teaches exact ATT&CK IDs (e.g. Kerberoasting = **T1558.003**, not the hallucinated T1060). This is **exp-002**, the project's first *real* fine-tune (`gemma3-cyber:v0.1` was only a system-prompt alias of the base model).

### Hardware Requirement
* Free Google Colab **T4 GPU** (15GB VRAM) or **L4/A100**. Sequence length 1024 keeps a T4 comfortable.

### Gemma-3 chat-format note
Gemma templates historically reject a standalone `system` role and use the role name `model` (not `assistant`). This notebook renders the training text with the repo's verified `to_gemma_chat_text()` (folds system into the first user turn) instead of relying on the tokenizer template, and masks the prompt so loss is computed only on the `model` turns.

## Step 1: Install Dependencies & Check GPU

In [ ]:
# Step 1: GPU check + PINNED dependency install.
#
# The FIRST exp-002 run installed this stack UNPINNED. The resolved `peft` required
# torchao > 0.16.0 while Colab shipped torchao 0.10.0, which crashed the merge step
# (Step 5). So we pin the stack and put `torchao>=0.16.0` in UP FRONT. These pins
# mirror configs/training/requirements-train.txt (the maintained source of truth).
!nvidia-smi
!pip install -q \
  "torch>=2.4,<2.9" "transformers>=4.50,<5" "peft>=0.13,<0.18" "trl>=0.12,<0.24" \
  "bitsandbytes>=0.44" "accelerate>=0.34" "datasets>=2.20" "torchao>=0.16.0" \
  "huggingface_hub>=0.24" "pyyaml>=6.0" "sentencepiece>=0.2" "protobuf>=4.25" "gguf>=0.10"

# Capture the EXACT resolved versions. This is the reproducibility fix: a `pip freeze`
# is persisted with the artifacts (Step 5.5) so the next run can pin `==` from it, and
# so we always know precisely what produced a given model.
import os
import subprocess
import sys

os.makedirs("/content/run_manifest", exist_ok=True)
_freeze = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True
).stdout
with open("/content/run_manifest/pip_freeze.txt", "w") as _f:
    _f.write(_freeze)

_key = {
    "torch", "transformers", "peft", "trl", "bitsandbytes",
    "accelerate", "datasets", "torchao", "gguf", "huggingface-hub",
}
print("Resolved key package versions for this run:")
for _line in _freeze.splitlines():
    if _line.split("==")[0].strip().lower() in _key:
        print("  ", _line)


## Step 1.5: Authenticate with Hugging Face

`google/gemma-3-4b-it` is a **gated** model, so the download needs an authenticated HF token.

1. **One-time access grant:** open [huggingface.co/google/gemma-3-4b-it](https://huggingface.co/google/gemma-3-4b-it) while logged in and click **"Agree and access repository"**. Wait until the page shows you have access (the Gemma gate can take minutes–hours).
2. **Create a READ token:** [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
3. **Store it in Colab:** click the 🔑 (Secrets) icon in the left sidebar → add a secret named `HF_TOKEN` with your token → enable **Notebook access**. (If you skip this, the cell below falls back to an interactive prompt.)

In [ ]:
# Authenticate so the gated Gemma download is authorized.
# Prefers a Colab secret named HF_TOKEN; falls back to an interactive prompt.
from huggingface_hub import login, whoami

token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    pass

login(token=token)  # token=None -> interactive widget to paste your HF token
print('Authenticated as:', whoami()['name'])


## Step 2: Clone Gemma4-CyberAi Repository & Validate Dataset

In [ ]:
import json, sys
from pathlib import Path

# If running in Colab without git clone, clone the repo (also puts src/ on the path).
dataset_path = 'data/training/sft_v0.2.jsonl'
if not Path(dataset_path).exists():
    print('Cloning repository...')
    !git clone https://github.com/novrusshehaj/Gemma4-CyberAi.git
    %cd Gemma4-CyberAi

# Make the package importable so we can reuse the VERIFIED Gemma formatter.
sys.path.insert(0, str(Path('src').resolve()))

# Verify dataset
with open(dataset_path, 'r', encoding='utf-8') as f:
    items = [json.loads(line) for line in f if line.strip()]
print(f'Successfully loaded {len(items)} training examples from {dataset_path}.')
assert len({json.dumps(it["id"]) for it in items}) == len(items), 'duplicate ids!'

## Step 3: Load Base Model with 4-bit Quantization (QLoRA)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = 'google/gemma-3-4b-it'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 4: Train LoRA Adapter with SFTTrainer

In [ ]:
import torch
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

# Use the repo's SHARED, UNIT-TESTED training helpers so this notebook and
# scripts/train_qlora.py format + mask IDENTICALLY (see gemma_cyber/training/sft.py).
# format_for_sft renders Gemma-3 turns with to_gemma_chat_text (folds `system` into
# the first user turn; emits `model` turns); build_completion_only_collator masks
# every token up to and including the first `<start_of_turn>model\n` so loss is
# computed only on the model completion.
from gemma_cyber.training import (
    build_completion_only_collator,
    format_for_sft,
    make_sft_config_kwargs,
    make_trainer_kwargs,
)

formatted = [format_for_sft(it["messages"]) for it in items]
train_dataset = Dataset.from_list(formatted)
print("Sample formatted example:\n", formatted[0]["text"][:300], "...")

collator = build_completion_only_collator(tokenizer)

base_cfg_kwargs = {
    "output_dir": "./results_gemma3_cyber_v0.2",
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "gradient_checkpointing": True,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "optim": "paged_adamw_8bit",
    "logging_steps": 10,
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "bf16": torch.cuda.is_bf16_supported(),
    "fp16": not torch.cuda.is_bf16_supported(),
    "seed": 42,
    "report_to": "none",
    "dataset_text_field": "text",
    "packing": False,
}
# make_sft_config_kwargs maps seq-length to max_length/max_seq_length by TRL version
# and drops any field this TRL doesn't know, so the cell is version-robust.
sft_config = SFTConfig(**make_sft_config_kwargs(SFTConfig, base_cfg_kwargs, 1024))

trainer = SFTTrainer(
    **make_trainer_kwargs(
        SFTTrainer,
        model=model,
        args=sft_config,
        train_dataset=train_dataset,
        data_collator=collator,
        tokenizer=tokenizer,
        peft_config=lora_config,
    )
)

trainer.train()
trainer.model.save_pretrained("./final_adapter")
tokenizer.save_pretrained("./final_adapter")
print("Training complete! Adapter saved to ./final_adapter")


## Step 5: Merge LoRA Adapter & Export to GGUF

In [ ]:
# Step 5 is SELF-CONTAINED: it re-imports and reloads everything it needs, so a Colab
# kernel restart between Step 4 and here does NOT break it (the first run hit exactly
# this: a restart wiped `AutoModelForCausalLM`/`torch`/`tokenizer` while `final_adapter`
# survived on disk). torchao>=0.16.0 is reasserted for the same reason as Step 1.
get_ipython().system('pip install -q -U "torchao>=0.16.0"')

import os
import subprocess

if os.path.isdir("/content/Gemma4-CyberAi") and os.path.basename(os.getcwd()) != "Gemma4-CyberAi":
    os.chdir("/content/Gemma4-CyberAi")
os.makedirs("/content/run_manifest", exist_ok=True)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "google/gemma-3-4b-it"
tokenizer = globals().get("tokenizer") or AutoTokenizer.from_pretrained(
    model_id, trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Merge the LoRA adapter into the base model in FP16.
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="cpu", trust_remote_code=True
)
merged_model = PeftModel.from_pretrained(base_model, "./final_adapter")
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained("./gemma3-cyber-v0.2-merged")
tokenizer.save_pretrained("./gemma3-cyber-v0.2-merged")
print("Merged model saved.")

# Convert to GGUF, then quantize to Q4_K_M for local Ollama serving. We RECORD the
# resolved llama.cpp commit so the conversion is reproducible; to reproduce a past run
# exactly, uncomment the checkout and paste the commit from that run's manifest.
if not os.path.isdir("llama.cpp"):
    get_ipython().system('git clone https://github.com/ggerganov/llama.cpp')
# get_ipython().system('cd llama.cpp && git checkout <PASTE_COMMIT_FROM_run_manifest>')
_llama_commit = subprocess.run(
    ["git", "-C", "llama.cpp", "rev-parse", "HEAD"], capture_output=True, text=True
).stdout.strip()
with open("/content/run_manifest/llama_cpp_commit.txt", "w") as _f:
    _f.write(_llama_commit + "\n")
print("llama.cpp commit:", _llama_commit)

get_ipython().system('pip install -q -r llama.cpp/requirements.txt')
get_ipython().system('python llama.cpp/convert_hf_to_gguf.py ./gemma3-cyber-v0.2-merged --outfile gemma3-cyber-v0.2.gguf --outtype f16')
get_ipython().system('cd llama.cpp && make -j llama-quantize')
get_ipython().system('./llama.cpp/llama-quantize gemma3-cyber-v0.2.gguf gemma3-cyber-v0.2-Q4_K_M.gguf Q4_K_M')

# VERIFY the export is real before claiming success (a print is not proof — the first
# run's file vanished on a disk reset). This checks existence, a plausible size, a
# SHA-256, and GGUF metadata (incl. the benign Gemma-3 "Duplicated key name" check),
# writing /content/run_manifest/verify_gguf.json. Non-zero exit -> a real problem.
_rc = subprocess.run(
    ["python", "scripts/verify_gguf_export.py", "gemma3-cyber-v0.2-Q4_K_M.gguf",
     "--out", "/content/run_manifest", "--min-size-mb", "1500"]
).returncode
assert _rc == 0, "GGUF export verification FAILED — see run_manifest/verify_gguf.json"
print("Exported AND VERIFIED gemma3-cyber-v0.2-Q4_K_M.gguf. Now run Step 5.5 to persist it.")


## Step 5.5: Persist artifacts (do NOT skip)

Colab's `/content` disk is **ephemeral** — a runtime reset wipes it. The first exp-002 run
trained and exported successfully, then lost every artifact to a reset. This step writes a
**run manifest** (versions, commits, checksums, config, dataset hash) and copies the GGUF +
adapter + manifest to **Google Drive** so they survive. Prefer Drive (free, durable); the
Hugging Face Hub upload at the bottom is an optional off-Colab backup.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
from datetime import datetime, timezone

os.makedirs("/content/run_manifest", exist_ok=True)


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def _git(*args):
    try:
        return subprocess.run(["git", *args], capture_output=True, text=True).stdout.strip()
    except Exception:
        return None


gguf_path = "gemma3-cyber-v0.2-Q4_K_M.gguf"
adapter_path = "final_adapter/adapter_model.safetensors"

manifest = {
    "experiment": "exp-002-gemma3-cyber-v0.2",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "repo_commit": _git("rev-parse", "HEAD"),
    "dataset": "data/training/sft_v0.2.jsonl",
    "dataset_sha256": _sha256("data/training/sft_v0.2.jsonl"),
    "training_config": "configs/training/qlora_gemma3_4b_v0.2.yaml",
    "seed": 42,
    "gguf": gguf_path,
    "gguf_sha256": _sha256(gguf_path) if os.path.exists(gguf_path) else None,
    "gguf_size_bytes": os.path.getsize(gguf_path) if os.path.exists(gguf_path) else None,
    "adapter_sha256": _sha256(adapter_path) if os.path.exists(adapter_path) else None,
}
with open("/content/run_manifest/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print(json.dumps(manifest, indent=2))

# --- Persist to Google Drive (durable across runtime resets) ---
try:
    from google.colab import drive

    drive.mount("/content/drive")
    dest = "/content/drive/MyDrive/gemma-cyber/exp-002"
    os.makedirs(dest, exist_ok=True)
    for item in [gguf_path, "final_adapter", "/content/run_manifest"]:
        if os.path.exists(item):
            target = os.path.join(dest, os.path.basename(item.rstrip("/")))
            if os.path.isdir(item):
                shutil.copytree(item, target, dirs_exist_ok=True)
            else:
                shutil.copy2(item, target)
    print(f"Persisted artifacts to {dest}")
    print("Verify they are there BEFORE closing the runtime:", os.listdir(dest))
except Exception as e:  # noqa: BLE001
    print("Drive persistence skipped/failed:", e)
    print("FALLBACK — download the GGUF + manifest to your machine now:")
    try:
        from google.colab import files

        files.download(gguf_path)
        files.download("/content/run_manifest/manifest.json")
    except Exception as e2:  # noqa: BLE001
        print("Manual download also unavailable:", e2)

# --- OPTIONAL off-Colab backup: push the GGUF to a (private) HF Hub repo ---
# from huggingface_hub import HfApi
# HfApi().upload_file(
#     path_or_fileobj=gguf_path, path_in_repo=gguf_path,
#     repo_id="<your-username>/gemma3-cyber-v0.2-gguf", repo_type="model",
# )


## Step 6: Create the Ollama model **on your own computer** and evaluate

Everything below runs on **your local machine** (where Ollama is installed), **not** in the
Colab terminal — Colab has no Ollama, and running `ollama create` there just prints
`ollama: command not found`. First download `gemma3-cyber-v0.2-Q4_K_M.gguf` from Google Drive
(or Colab) to the repo checkout on your machine, then:

```bash
# 1. Re-verify the file survived the trip off Colab (compare against run_manifest/manifest.json).
python scripts/verify_gguf_export.py gemma3-cyber-v0.2-Q4_K_M.gguf --min-size-mb 1500
#    -> sha256 must match manifest.json's gguf_sha256.

# 2. Point an Ollama Modelfile's FROM line at the v0.2 GGUF, then create the model.
#    (Copy Modelfile.template, change FROM to ./gemma3-cyber-v0.2-Q4_K_M.gguf.)
ollama create gemma3-cyber:v0.2 -f Modelfile.template
ollama run gemma3-cyber:v0.2 "Explain the MITRE ATT&CK technique for Kerberoasting."
```

Then run the **pre-registered** evaluation — base vs. v0.2 on the frozen v2 anchor **and** the
targeted v3 instrument, scoring the held-out `test` split for the final comparison:

```bash
# do-no-harm / regression anchor (v2)
python scripts/run_baseline.py --model gemma3-cyber:v0.2 \
    --benchmark data/evaluation/benchmark_v2.jsonl --split test \
    --out experiments/exp-002-gemma3-cyber-v0.2/v0.2-v2-test
# targeted sensitivity instrument (v3) — ATT&CK precision, false premises, factual scorer
python scripts/run_baseline.py --model gemma3-cyber:v0.2 \
    --benchmark data/evaluation/benchmark_v3.jsonl --split test \
    --out experiments/exp-002-gemma3-cyber-v0.2/v0.2-v3-test
```

Compare against the base `gemma3:4b` run on the same benchmarks/splits (Step 6 commands in
`docs/experiments/exp-002.md`). The candidate's pass/fail is decided **only** by those
scorecards against the success criteria in `docs/experiments/exp-002.md` §4 — training loss
is not evidence of model quality. See `docs/training/README.md` for the full reproduce +
deploy runbook.